# Can Fitness-Browser data generalize? — same-organism cross-condition first

Tests whether a learned model on **dense per-organism fitness data** generalizes within an organism (cross-condition) before we ask if it crosses organism boundaries. The Ralstonia attempt long ago had only 7 conditions (degenerate); this notebook works on the organisms with hundreds.

**Setup**: download `feba.db` (~2.3 GB compressed, ~7 GB uncompressed), filter to organisms with >=50 conditions, build the (gene, condition) -> fitness matrix, fit a small MLP that takes per-gene phylogenetic features + a condition embedding, predict held-out CONDITIONS first.

**What 'generalize' means here**: train on a subset of conditions for an organism, predict fitness on conditions the model never saw. Strong result = ρ ≥ 0.4 (Tn-seq's own continuous noise floor is ρ ≈ 0.38, ceiling ρ ≈ 0.77).

**Next session** (after this works): leave-one-organism-out, the harder test.

**Storage**: ~10 GB total. Fits any Colab. CPU is enough; GPU only speeds the small MLP.

## 1. Install + clone repo + download feba.db

In [ ]:
!pip install -q pandas numpy scipy scikit-learn
!git clone --depth 1 -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git cell_repo || echo cloned
import os; os.chdir('cell_repo')
# pull feba.db via the existing kappa_floor downloader (handles figshare resolve)
import sys; sys.path.insert(0,'scripts')
from pathlib import Path
from kappa_floor import fetch_feba
FEBA = Path('/tmp/feba.db')
if not FEBA.exists():
    fetch_feba(FEBA)
!ls -lh {FEBA}

## 2. Inspect schema; find dense organisms

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect('/tmp/feba.db')
print('tables:', pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con).name.tolist())
exp = pd.read_sql('SELECT orgId, expName, expGroup, condition_1, media, aerobic FROM Experiment', con)
print(f'experiments total: {len(exp):,} | organisms: {exp.orgId.nunique()}')
dense = exp.groupby('orgId').size().sort_values(ascending=False)
print('\norganisms with most experiments:')
print(dense.head(15).to_string())
print(f'\norganisms with >=50 conditions: {(dense>=50).sum()}')
DENSE_ORGS = dense[dense>=50].index.tolist()

## 3. Build the (gene, condition) tensor — focal organism first
Start with the organism that has the most conditions. We'll build a single (gene, condition) table with fitness `fit` and t-statistic `t`.

In [ ]:
FOCAL = DENSE_ORGS[0]
print(f'focal organism: {FOCAL} ({dense[FOCAL]} experiments)')
# pull this org's fitness rows (the table is wide -- many rows per gene*expt)
gf = pd.read_sql(f"SELECT * FROM GeneFitness WHERE orgId='{FOCAL}'", con)
print('GeneFitness columns:', gf.columns.tolist())
print('shape:', gf.shape)
print(gf.head(3).to_string())
print(f'\nunique experiments: {gf.expName.nunique() if "expName" in gf else "?"}')
print(f'unique genes (locusId): {gf.locusId.nunique() if "locusId" in gf else "?"}')

## 4. Quick reality check — how reproducible is the focal organism's data?
Pick same-condition experiment pairs, compute Spearman of fitness across genes. This is the noise floor for this organism specifically — and what any predictor of these labels must beat to be real.

In [ ]:
from scipy.stats import spearmanr
# join experiments with condition info
ex_f = exp[exp.orgId==FOCAL].copy()
ex_f['cluster'] = ex_f.media.fillna('-')+'|'+ex_f.aerobic.fillna('-')+'|'+ex_f.condition_1.fillna('-')
print(f'condition clusters: {ex_f.cluster.nunique()}  (median experiments per cluster: {ex_f.groupby("cluster").size().median():.0f})')
rhos = []
for cl, sub in ex_f.groupby('cluster'):
    if len(sub)<2: continue
    names = sub.expName.tolist()
    for i in range(len(names)):
        for j in range(i+1, min(i+3, len(names))):       # cap pairs
            a = gf[gf.expName==names[i]].set_index('locusId').fit
            b = gf[gf.expName==names[j]].set_index('locusId').fit
            common = a.index.intersection(b.index)
            if len(common)<200: continue
            rhos.append(spearmanr(a.loc[common], b.loc[common]).statistic)
import numpy as np
rhos = [r for r in rhos if r==r]
print(f'same-cluster Spearman pairs: n={len(rhos)}, median={np.median(rhos):.3f}, '
      f'10/50/90%: {np.percentile(rhos,10):.2f}/{np.median(rhos):.2f}/{np.percentile(rhos,90):.2f}')
print('-> THIS is the ceiling any condition-generalizing predictor can hit.')

## 5. Build features for the cross-condition prediction
Two halves of the input:
- **gene features**: log-length, GC, conservation/family_frac, n_paralogs (from `orthology_features.csv` if the focal organism is in our labeled set; otherwise just length+GC from `feba.db`'s `Gene` table)
- **condition features**: one-hot of (media, stress) at first — start simple. Held-out conditions must SHARE at least one feature dimension with seen ones.

In [ ]:
import numpy as np, pandas as pd
# gene features
g = pd.read_sql(f"SELECT * FROM Gene WHERE orgId='{FOCAL}'", con)
print('Gene columns:', g.columns.tolist())
g['log_len'] = np.log1p((g.end-g.begin+1).clip(lower=1)/3)
g_feat = g.set_index('locusId')[['log_len']].copy()
g_feat['gc'] = 0.5   # placeholder if no GC col; refine later
# condition features (media + condition_1 + aerobic as categorical one-hots)
ex_f['key'] = ex_f.media.fillna('?')+'__'+ex_f.condition_1.fillna('none')+'__'+ex_f.aerobic.fillna('?')
media_cat = ex_f.media.fillna('?').astype('category').cat.codes.values
cond_cat  = ex_f.condition_1.fillna('none').astype('category').cat.codes.values
aer_cat   = ex_f.aerobic.fillna('?').astype('category').cat.codes.values
n_med = int(media_cat.max())+1; n_cond = int(cond_cat.max())+1; n_aer = int(aer_cat.max())+1
print(f'unique media: {n_med} | unique condition_1: {n_cond} | aerobic: {n_aer}')
ex_f = ex_f.assign(media_i=media_cat, cond_i=cond_cat, aer_i=aer_cat)
print('focal experiments:', len(ex_f))

## 6. Build the (gene, condition) -> fit training tensor + LOO-condition split
**Hold-out rule**: pick a condition_1 value, hold out *every* experiment whose `condition_1` matches. The model never sees that stress at training. This is the right cross-condition generalization test.

In [ ]:
# z-normalize fit per-experiment so the loss isn't dominated by experiment-level scale
gf2 = gf.merge(ex_f[['expName','media_i','cond_i','aer_i','condition_1']], on='expName', how='inner')
gf2 = gf2.merge(g_feat, left_on='locusId', right_index=True)
gf2['fit_z'] = gf2.groupby('expName').fit.transform(lambda x: (x - x.median())/(x.std()+1e-6))
print('training pairs:', len(gf2))
print(gf2[['locusId','expName','fit','fit_z','log_len','media_i','cond_i','aer_i','condition_1']].head(3).to_string())

# build LOO-condition folds over condition_1 values seen in >=2 experiments
fold_keys = ex_f.condition_1.value_counts()
fold_keys = fold_keys[fold_keys>=2].index.tolist()[:8]   # at most 8 folds for speed
print(f'LOO-condition folds: {len(fold_keys)} -> {fold_keys[:6]}...')

## 7. Train a small MLP per fold, report Spearman vs the noise floor
**Architecture**: gene one-hot embedding (E_g, learned) concatenated with one-hots(media, cond_1, aerobic) plus the gene's `log_len`. Output = z-normalized fit. Tiny MLP. Pure numpy with Adam for portability.

In [ ]:
import numpy as np
from scipy.stats import spearmanr
# build integer indices for gene one-hot (embedded later)
gene_ids = gf2.locusId.astype('category')
gf2 = gf2.assign(gene_i=gene_ids.cat.codes)
n_gene = int(gf2.gene_i.max())+1
print(f'n_gene={n_gene}, n_med={n_med}, n_cond={n_cond}, n_aer={n_aer}')

E_G=32; E_C=16; HID=64
def init(rng):
    return dict(
        eg=rng.normal(0,.3,(n_gene,E_G)),
        em=rng.normal(0,.3,(n_med,E_C)),
        ec=rng.normal(0,.3,(n_cond,E_C)),
        ea=rng.normal(0,.3,(n_aer,E_C)),
        W1=rng.normal(0,.3,(E_G+3*E_C+1,HID)), b1=np.zeros(HID),
        w2=rng.normal(0,.3,HID), b2=0.0)
def fwd(P, gi, mi, ci, ai, ll):
    x = np.concatenate([P['eg'][gi], P['em'][mi], P['ec'][ci], P['ea'][ai], ll[:,None]], axis=1)
    h = np.maximum(x@P['W1']+P['b1'],0); return (h@P['w2']+P['b2']), x, h

fold_rho = []
for held in fold_keys:
    tr = gf2[gf2.condition_1 != held]; te = gf2[gf2.condition_1 == held]
    if len(te)<200: continue
    rng = np.random.default_rng(0); P = init(rng)
    M={k:0 for k in P}; V=dict(M); t=0; lr=3e-3
    gi=tr.gene_i.to_numpy(); mi=tr.media_i.to_numpy(); ci=tr.cond_i.to_numpy(); ai=tr.aer_i.to_numpy()
    ll=tr.log_len.to_numpy(); yt=tr.fit_z.to_numpy()
    n=len(tr); bs=2048
    for ep in range(5):
        idx = rng.permutation(n)
        for s in range(0, n, bs):
            b = idx[s:s+bs]
            yh, x, h = fwd(P, gi[b], mi[b], ci[b], ai[b], ll[b])
            err = (yh - yt[b]) / len(b)
            gw2 = h.T@err; gb2 = err.sum(); dh = np.outer(err,P['w2'])*(h>0)
            gW1 = x.T@dh; gb1 = dh.sum(0); dx = dh@P['W1'].T
            # embedding grads
            d_eg=dx[:,:E_G]; d_em=dx[:,E_G:E_G+E_C]; d_ec=dx[:,E_G+E_C:E_G+2*E_C]; d_ea=dx[:,E_G+2*E_C:E_G+3*E_C]
            grads={'W1':gW1,'b1':gb1,'w2':gw2,'b2':gb2}
            for emb,idxs,d,nm in [('eg',gi[b],d_eg,'eg'),('em',mi[b],d_em,'em'),('ec',ci[b],d_ec,'ec'),('ea',ai[b],d_ea,'ea')]:
                g = np.zeros_like(P[emb]); np.add.at(g, idxs, d); grads[emb]=g
            t+=1
            for k in grads:
                if not isinstance(M[k],np.ndarray): M[k]=np.zeros_like(grads[k]); V[k]=np.zeros_like(grads[k])
                M[k]=0.9*M[k]+0.1*grads[k]; V[k]=0.999*V[k]+0.001*(grads[k]**2)
                P[k] -= lr*(M[k]/(1-0.9**t))/(np.sqrt(V[k]/(1-0.999**t))+1e-8)
    # eval
    yh,_,_ = fwd(P, te.gene_i.to_numpy(), te.media_i.to_numpy(), te.cond_i.to_numpy(),
                 te.aer_i.to_numpy(), te.log_len.to_numpy())
    rho = spearmanr(yh, te.fit_z.to_numpy()).statistic
    fold_rho.append((held, len(te), rho))
    print(f'  fold {held[:50]:<50} n_test={len(te):>6}  rho={rho:.3f}')
print(f'\nCROSS-CONDITION Spearman (median across {len(fold_rho)} folds): '
      f'{np.median([r for _,_,r in fold_rho]):.3f}')

## 8. Interpret

Three reference points to compare the cross-condition ρ to:
- **Same-condition replicate ρ** (cell 4): the noise ceiling. Predictor cannot exceed this.
- **ρ ≈ 0.38** (median from Paper 1's 17,950 same-condition pairs across 48 orgs): generic Tn-seq noise.
- **ρ ≈ 0.77** (calibration-corrected ceiling).

Verdicts:
- ρ ≥ 0.4 cross-condition → condition generalization works in-organism. Worth trying cross-organism next.
- 0.2 ≤ ρ < 0.4 → modest, real but limited.
- ρ < 0.2 → condition-specific essentiality is ~unpredictable from a held-out condition. The condition embedding is the bottleneck — would need richer condition descriptors (chemical structure of stressor, gene-ontology of target, etc.).

Save the fold results back to the repo:

In [ ]:
import json, numpy as np
out = {'focal_org': FOCAL, 'noise_floor_median_rho': float(np.median(rhos)),
       'fold_results': [{'condition_held': k, 'n_test': int(n), 'rho': float(r)} for k,n,r in fold_rho],
       'cross_condition_median_rho': float(np.median([r for _,_,r in fold_rho])),
       'n_dense_orgs_available': len(DENSE_ORGS)}
json.dump(out, open('outputs/orphan/fitness_generalize_within.json','w'), indent=2)
print(json.dumps(out, indent=2))